# 01 — EDA и предобработка данных

## Датасет: Ru-GoEmotions

**Ru-GoEmotions** — русскоязычная версия датасета GoEmotions от Google (Демидовский и др.).
Содержит 28 тонких эмоциональных категорий. Мы группируем их в **7 классов по таксономии Ekman**:

| ID | Класс | Исходные эмоции |
|---|---|---|
| 0 | anger | anger, annoyance, disapproval |
| 1 | disgust | disgust |
| 2 | fear | fear, nervousness |
| 3 | joy | admiration, amusement, approval, caring, desire, excitement, gratitude, joy, love, optimism, pride, relief |
| 4 | sadness | disappointment, embarrassment, grief, remorse, sadness |
| 5 | surprise | confusion, curiosity, realization, surprise |
| 6 | neutral | neutral |

In [ ]:
import sys, os

PROJECT_ROOT = "/kaggle/input/sentiment-analysis" if os.path.exists("/kaggle") else os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "../results"
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working dir: {WORKING_DIR}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120

## 1. Загрузка данных

In [ ]:
from src.data_loader import load_data, EKMAN_LABEL_NAMES, EKMAN_ID2LABEL

# load_data() автоматически скачает Ru-GoEmotions и применит Ekman-группировку
# Если у вас есть свой CSV: load_data(csv_path='/path/to/file.csv')
dataset, label_names, info = load_data()

print(f'Классы ({len(label_names)}): {label_names}')
print('\nСтатистика:')
for split, stat in info.items():
    print(f'  {split:12s}: {stat["size"]:>6,} примеров')
    for cls, cnt in sorted(stat['label_distribution'].items()):
        print(f'    {cls:<12s}: {cnt:>5,}')

In [ ]:
train_df = dataset['train'].to_pandas()
val_df   = dataset['validation'].to_pandas()
test_df  = dataset['test'].to_pandas()
all_df   = pd.concat([train_df, val_df, test_df], ignore_index=True)
all_df['emotion'] = all_df['label'].map(EKMAN_ID2LABEL)
print(f'Всего: {len(all_df):,} примеров')
all_df.head()

## 2. Распределение эмоций

In [ ]:
EMOTION_COLORS = {
    'anger':   '#e74c3c',
    'disgust': '#8e44ad',
    'fear':    '#2c3e50',
    'joy':     '#f39c12',
    'sadness': '#3498db',
    'surprise':'#1abc9c',
    'neutral': '#95a5a6',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Абсолютные числа
counts = all_df['emotion'].value_counts().reindex(label_names)
colors = [EMOTION_COLORS[e] for e in counts.index]
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title('Распределение классов (всего)')
axes[0].set_xlabel('Эмоция')
axes[0].set_ylabel('Количество')
for i, (cls, v) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, v + 30, f'{v:,}\n({v/len(all_df)*100:.1f}%)', ha='center', fontsize=8)

# По сплитам — нормализованные
split_data = {}
for split, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    split_data[split] = df['label'].map(EKMAN_ID2LABEL).value_counts(normalize=True).reindex(label_names).fillna(0)
pd.DataFrame(split_data).plot(kind='bar', ax=axes[1], color=['#3498db', '#e67e22', '#9b59b6'])
axes[1].set_title('Распределение по сплитам (доли)')
axes[1].set_xticklabels(label_names, rotation=30, ha='right')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/emotion_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Длины текстов по классам
all_df['word_count'] = all_df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot длин по эмоциям
data_by_emotion = [all_df[all_df['emotion'] == e]['word_count'].dropna().values for e in label_names]
axes[0].boxplot(data_by_emotion, labels=label_names, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[0].set_title('Длина текстов по эмоциям (кол-во слов)')
axes[0].set_xlabel('Эмоция')
axes[0].set_ylabel('Кол-во слов')
axes[0].tick_params(axis='x', rotation=30)

# Гистограмма длин
axes[1].hist(all_df['word_count'], bins=50, color='#3498db', alpha=0.7, edgecolor='white')
axes[1].axvline(all_df['word_count'].median(), color='red', linestyle='--', label=f'Медиана: {all_df["word_count"].median():.0f}')
axes[1].set_title('Распределение длин текстов')
axes[1].set_xlabel('Кол-во слов')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/text_lengths.png', dpi=150, bbox_inches='tight')
plt.show()
print(all_df[['word_count']].describe().round(1))

In [ ]:
# Примеры текстов по классам
for emotion in label_names:
    print(f'\n{"═"*60}')
    print(f'  {emotion.upper()}')
    print(f'{"═"*60}')
    samples = all_df[all_df['emotion'] == emotion]['text'].sample(
        min(2, (all_df['emotion'] == emotion).sum()), random_state=42
    )
    for i, text in enumerate(samples, 1):
        print(f'{i}. {text[:180]}...' if len(text) > 180 else f'{i}. {text}')

## 3. Предобработка

In [ ]:
from src.preprocessor import clean_text, preprocess_batch

# Примеры очистки
test_texts = [
    'Ужасный день!!! Всё пошло не так 😭 #грусть http://example.com',
    '@друг Это просто невероятно!! Я так счастлив!!!',
    'Не знаю что и думать... наверное нормально',
]
for text in test_texts:
    print(f'  До:   {text}')
    print(f'  После: {clean_text(text)}')
    print()

In [ ]:
print('Применяю предобработку...')
dataset_clean = dataset.map(
    lambda batch: preprocess_batch(batch, text_column='text', clean=True),
    batched=True, batch_size=1000, desc='Cleaning',
)
print('Готово!')

## 4. Сохранение

In [ ]:
import json

# Сохраняем датасет
DATA_SAVE_PATH = f'{WORKING_DIR}/processed_data'
dataset_clean.save_to_disk(DATA_SAVE_PATH)

# Сохраняем label_names для использования в следующих ноутбуках
with open(f'{WORKING_DIR}/label_names.json', 'w') as f:
    json.dump(label_names, f)

# CSV для удобства
for split in ['train', 'validation', 'test']:
    df = dataset_clean[split].to_pandas()
    df['emotion'] = df['label'].map(EKMAN_ID2LABEL)
    df.to_csv(f'{WORKING_DIR}/{split}.csv', index=False, encoding='utf-8')
    print(f'{split}.csv: {len(df):,} строк')

print(f'\nВсё сохранено в: {WORKING_DIR}')